# public_LPS_data_Emily.ipynb

In [1]:
import gdown

In [3]:
url = 'https://ftp.ebi.ac.uk/biostudies/fire/E-MTAB-/026/E-MTAB-10026/Files/covid_portal_210320_with_raw.h5ad'
output= './Raw_h5ad.zip'
gdown.download(url,output)

Downloading...
From: https://ftp.ebi.ac.uk/biostudies/fire/E-MTAB-/026/E-MTAB-10026/Files/covid_portal_210320_with_raw.h5ad
To: /home/jovyan/farm_mount/Cytomeister/Evaluation_datasets/LPS/Public_Emily/Raw_h5ad.zip
100%|██████████| 7.19G/7.19G [01:31<00:00, 78.9MB/s]


'./Raw_h5ad.zip'

In [1]:
import scanpy as sc

In [21]:
adata = sc.read('./Raw.h5ad')

In [22]:
adata

AnnData object with n_obs × n_vars = 647366 × 24929
    obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'full_clustering', 'initial_clustering', 'Resample', 'Collection_Day', 'Sex', 'Age_interval', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'patient_id'
    var: 'feature_types'
    uns: 'hvg', 'leiden', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    layers: 'raw'

In [23]:
adata.obs['time_after_LPS'].value_counts()

time_after_LPS
nan    639482
90m      3999
10h      3885
Name: count, dtype: int64

In [24]:
adata.obs['Status'].value_counts()

Status
Covid        527286
Healthy       97039
Non_covid     15157
LPS            7884
Name: count, dtype: int64

In [25]:
adata

AnnData object with n_obs × n_vars = 647366 × 24929
    obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'full_clustering', 'initial_clustering', 'Resample', 'Collection_Day', 'Sex', 'Age_interval', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'patient_id'
    var: 'feature_types'
    uns: 'hvg', 'leiden', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    layers: 'raw'

In [26]:
adata.obs['Status'].value_counts()

Status
Covid        527286
Healthy       97039
Non_covid     15157
LPS            7884
Name: count, dtype: int64

In [27]:
lps_sample_ids = adata.obs.loc[adata.obs['Status'] == 'LPS', 'patient_id'].unique()

healthy_sample_ids = adata.obs.loc[adata.obs['Status'] == 'Healthy', 'patient_id'].unique()


In [28]:
adata = adata[adata.obs['Status'].isin(['LPS','Healthy'])].copy()

In [29]:
import pandas as pd
adata.obs['time_after_LPS'] = adata.obs['time_after_LPS'].astype(str)
adata.obs['time_after_LPS'] = adata.obs['time_after_LPS'].replace('nan', 'normal')



In [30]:
adata.obs['time_after_LPS'] = adata.obs['time_after_LPS'].apply(lambda x: x if x == 'normal' else f'{x}_LPS')

adata.obs['time_after_LPS'] = pd.Categorical(
    adata.obs['time_after_LPS'],
    categories=['normal', '90m_LPS', '10h_LPS'],
    ordered=True
)

In [31]:
adata.obs['time_after_LPS'].value_counts()

time_after_LPS
normal     97039
90m_LPS     3999
10h_LPS     3885
Name: count, dtype: int64

In [33]:
adata

AnnData object with n_obs × n_vars = 104923 × 24929
    obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'full_clustering', 'initial_clustering', 'Resample', 'Collection_Day', 'Sex', 'Age_interval', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'patient_id', 'donor_id'
    var: 'feature_types'
    uns: 'hvg', 'leiden', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    layers: 'raw'

In [32]:
adata.obs['donor_id'] = adata.obs['patient_id']

In [37]:
adata.obs['study'] = 'Public_Emily2021'

In [36]:
adata.obs['Age_interval'].value_counts()

Age_interval
(30, 39]    30538
(50, 59]    26344
(20, 29]    20426
(40, 49]    15926
(60, 69]     9970
(70, 79]     1719
Name: count, dtype: int64

In [38]:
mapping = {
    '(20, 29]': 'Adult',      
    '(30, 39]': 'Adult',
    '(40, 49]': 'Adult',
    '(50, 59]': 'Adult',
    '(60, 69]': 'Adult',
    '(70, 79]': 'Old'
}


In [39]:
adata.obs['life_stage'] = adata.obs['Age_interval'].map(mapping)


adata.obs['life_stage'] = pd.Categorical(
    adata.obs['life_stage'],
    categories=['Embryo', 'Fetal', 'Childhood', 'Young Adult', 'Adult', 'Old'],
    ordered=True
)


In [40]:
adata.obs['development_stage'] = adata.obs['life_stage']

In [41]:
adata.obs['tissue'] = 'blood'

In [45]:
adata.X = adata.layers['raw'].copy()

In [46]:
adata.X.max()

np.float32(275279.0)

In [49]:
adata

AnnData object with n_obs × n_vars = 104923 × 24929
    obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'full_clustering', 'initial_clustering', 'Resample', 'Collection_Day', 'Sex', 'Age_interval', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'patient_id', 'donor_id', 'study', 'life_stage', 'development_stage', 'tissue'
    var: 'feature_types'
    uns: 'hvg', 'leiden', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    layers: 'raw'

In [51]:
adata.obs['Resample']

covid_index
AAACCTGAGACCACGA-newcastle65    Initial
AAACCTGAGATGTCGG-newcastle65    Initial
AAACCTGAGGCGATAC-newcastle65    Initial
AAACCTGAGTACACCT-newcastle65    Initial
AAACCTGAGTGAATTG-newcastle65    Initial
                                 ...   
BGCV15_TTTCCTCTCTGATACG-1       Initial
BGCV15_TTTCCTCTCTTTAGGG-1       Initial
BGCV15_TTTGCGCTCACCGTAA-1       Initial
BGCV15_TTTGGTTTCAAGATCC-1       Initial
BGCV15_TTTGTCACAAGCCATT-1       Initial
Name: Resample, Length: 104923, dtype: category
Categories (1, object): ['Initial']

In [52]:
# Subset to Gene Expression features
adata = adata[:, adata.var['feature_types'] == 'Gene Expression']


In [54]:
import pandas as pd
ref = pd.read_csv("../../../../Ensembl_symbol_Human_(GRCh38.p14)/mart_export.txt")  


In [56]:
adata.var

,feature_types
MIR1302-2HG,Gene Expression
AL627309.1,Gene Expression
AL627309.3,Gene Expression
AL627309.2,Gene Expression
AL669831.2,Gene Expression
...,...
AC007325.2,Gene Expression
AL354822.1,Gene Expression
AC233755.2,Gene Expression
AC233755.1,Gene Expression


In [57]:
# 1. Convert to string and clean
ref.index = ref.index.astype(str).str.strip().str.upper()
ref["HGNC symbol"] = ref["HGNC symbol"].astype(str).str.strip().str.upper()

# Drop NaNs in gene_symbol (after converting to str 'nan', you can use replace)
ref = ref.replace({"HGNC symbol": {"NAN": None}}).dropna(subset=["HGNC symbol"])

# 2. Map gene_symbol -> ensembl_id by first occurrence
ref_rev = (
    ref.reset_index()
       .drop_duplicates(subset=["HGNC symbol"])
       .set_index("HGNC symbol")["Gene stable ID"]
)

# 3. Map gene_symbol from adata.var_names to ensembl_id
adata.var["gene_symbol"] = adata.var_names.str.strip().str.upper()
adata.var["ensembl_id"] = adata.var["gene_symbol"].map(ref_rev)

# 4. Filter genes without mapping
adata = adata[:, adata.var["ensembl_id"].notna()].copy()

# 5. Set var_names to ensembl_id
adata.var_names = adata.var["ensembl_id"]

print(f"adata now has {adata.n_vars} genes with Ensembl IDs as var_names.")


/tmp/ipykernel_962786/4272093940.py:16: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["gene_symbol"] = adata.var_names.str.strip().str.upper()


adata now has 19193 genes with Ensembl IDs as var_names.


In [60]:
adata.var.index.name = 'gene_id'


In [61]:
adata.var

,feature_types,gene_symbol,ensembl_id
gene_id,,,
ENSG00000243485,Gene Expression,MIR1302-2HG,ENSG00000243485
ENSG00000177757,Gene Expression,FAM87B,ENSG00000177757
ENSG00000225880,Gene Expression,LINC00115,ENSG00000225880
ENSG00000230368,Gene Expression,FAM41C,ENSG00000230368
ENSG00000187634,Gene Expression,SAMD11,ENSG00000187634
...,...,...,...
ENSG00000198886,Gene Expression,MT-ND4,ENSG00000198886
ENSG00000198786,Gene Expression,MT-ND5,ENSG00000198786
ENSG00000198695,Gene Expression,MT-ND6,ENSG00000198695


In [65]:
adata.obs['development_stage']


covid_index
AAACCTGAGACCACGA-newcastle65    Adult
AAACCTGAGATGTCGG-newcastle65    Adult
AAACCTGAGGCGATAC-newcastle65    Adult
AAACCTGAGTACACCT-newcastle65    Adult
AAACCTGAGTGAATTG-newcastle65    Adult
                                ...  
BGCV15_TTTCCTCTCTGATACG-1       Adult
BGCV15_TTTCCTCTCTTTAGGG-1       Adult
BGCV15_TTTGCGCTCACCGTAA-1       Adult
BGCV15_TTTGGTTTCAAGATCC-1       Adult
BGCV15_TTTGTCACAAGCCATT-1       Adult
Name: development_stage, Length: 104923, dtype: category
Categories (2, object): ['Adult' < 'Old']

In [70]:
adata.obs['full_clustering'].value_counts()

full_clustering
CD4.Naive                13829
NK_16hi                  13473
CD8.Naive                 8105
CD4.CM                    7505
B_naive                   6908
CD83_CD14_mono            6772
CD8.TE                    6501
CD4.IL22                  6379
CD8.EM                    5734
gdT                       4948
CD14_mono                 3774
MAIT                      3465
CD16_mono                 3007
Platelets                 2363
NK_56hi                   2190
B_switched_memory         1395
DC2                       1052
DC3                        975
B_immature                 966
NKT                        864
CD4.Tfh                    800
pDC                        678
B_non-switched_memory      647
B_exhausted                334
RBC                        301
NK_prolif                  221
ILC1_3                     213
C1_CD16_mono               198
CD4.EM                     188
Plasma_cell_IgA            173
Plasma_cell_IgG            168
DC1                    

In [71]:
adata.obs['cell_type'] = adata.obs['full_clustering'].copy()

In [72]:
cell_type_counts = adata.obs['cell_type'].value_counts()

valid_cell_types = cell_type_counts[cell_type_counts >= 50].index

adata = adata[adata.obs['cell_type'].isin(valid_cell_types)].copy()


In [73]:
adata.obs['time_after_LPS']

covid_index
AAACCTGAGACCACGA-newcastle65    normal
AAACCTGAGATGTCGG-newcastle65    normal
AAACCTGAGGCGATAC-newcastle65    normal
AAACCTGAGTACACCT-newcastle65    normal
AAACCTGAGTGAATTG-newcastle65    normal
                                 ...  
BGCV15_TTTCCTCTCTGATACG-1       normal
BGCV15_TTTCCTCTCTTTAGGG-1       normal
BGCV15_TTTGCGCTCACCGTAA-1       normal
BGCV15_TTTGGTTTCAAGATCC-1       normal
BGCV15_TTTGTCACAAGCCATT-1       normal
Name: time_after_LPS, Length: 104790, dtype: category
Categories (3, object): ['normal' < '90m_LPS' < '10h_LPS']

In [74]:
import numpy as np

# If your data is sparse, it's best to use .X.sum(axis=1).A1
if hasattr(adata.X, 'sum'):
    adata.obs['n_counts'] = np.ravel(adata.X.sum(axis=1))
else:
    adata.obs['n_counts'] = adata.X.sum(axis=1)


In [75]:
adata.obs['n_counts']

covid_index
AAACCTGAGACCACGA-newcastle65    4242.0
AAACCTGAGATGTCGG-newcastle65    4708.0
AAACCTGAGGCGATAC-newcastle65    2858.0
AAACCTGAGTACACCT-newcastle65    5227.0
AAACCTGAGTGAATTG-newcastle65    4012.0
                                 ...  
BGCV15_TTTCCTCTCTGATACG-1       5766.0
BGCV15_TTTCCTCTCTTTAGGG-1       7415.0
BGCV15_TTTGCGCTCACCGTAA-1       5168.0
BGCV15_TTTGGTTTCAAGATCC-1       6504.0
BGCV15_TTTGTCACAAGCCATT-1       5246.0
Name: n_counts, Length: 104790, dtype: float32

In [76]:
adata.write_h5ad('./Public_LPS_PP.h5ad')